# Teacher Source South Kensington Rollout-finetuned Surrogate Test

Run the rollout-finetuned temperature surrogate against South Kensington physical 3D temperature results. It runs two cases:

- `case_baseline_existing_source`: original South Kensington source distribution;
- `case_right_half_source`: same South Kensington geometry/velocity and baseline thermal condition, but only the right half keeps the existing source and the left half has no source.



In [ ]:
from pathlib import Path
import json
import shutil
import sys
from dataclasses import asdict

import numpy as np
import matplotlib.pyplot as plt
import torch
from IPython.display import Image as IPyImage, display

cwd = Path.cwd().resolve()
if (cwd / "test_virtual_geometry_rollout_surrogate.py").exists():
    ROLLOUT_DIR = cwd
elif (cwd / "rollout_surrogate" / "test_virtual_geometry_rollout_surrogate.py").exists():
    ROLLOUT_DIR = cwd / "rollout_surrogate"
else:
    raise FileNotFoundError(
        "Open this notebook from temperature_field_bundle or rollout_surrogate."
    )

if str(ROLLOUT_DIR) not in sys.path:
    sys.path.insert(0, str(ROLLOUT_DIR))

import test_virtual_geometry_rollout_surrogate as t

if str(t.BUNDLE_ROOT) not in sys.path:
    sys.path.insert(0, str(t.BUNDLE_ROOT))

from south_kensington_jupyter import SouthKensingtonConfig
from south_kensington_temperature_3d import (
    Temperature3DScenarioConfig,
    extract_or_load_full_velocity_fields,
    build_surface_temperature_fields,
    solve_temperature_fields_3d,
    save_results,
)

SK_CASE_DIR = t.BUNDLE_ROOT / "outputs" / "south_kensington_temperature_3d"
RIGHT_HALF_CASE_DIR = (
    t.BUNDLE_ROOT / "outputs" / "south_kensington_temperature_3d_right_half_source"
)
OUT_DIR = t.ROLLOUT_RUN_DIR / "south_kensington_rollout_test"
OUT_DIR.mkdir(parents=True, exist_ok=True)

RUN_RIGHT_HALF_PHYSICAL_CASE = True
FORCE_REGENERATE_RIGHT_HALF_PHYSICAL = False

CASE_SPECS = [
    {
        "case_id": "case_baseline_existing_source",
        "case_dir": SK_CASE_DIR,
        "source_mask_name": "existing_source",
        "note": "Original South Kensington physical case with the existing full source distribution.",
    },
    {
        "case_id": "case_right_half_source",
        "case_dir": RIGHT_HALF_CASE_DIR,
        "source_mask_name": "right_half_source",
        "note": "South Kensington geometry/velocity/baseline thermal condition with source only in the right half of the domain.",
    },
]
CASE_SPEC_BY_ID = {spec["case_id"]: spec for spec in CASE_SPECS}

print("Rollout dir:", ROLLOUT_DIR)
print("Bundle root:", t.BUNDLE_ROOT)
print("South Kensington baseline case:", SK_CASE_DIR)
print("Right-half source case:", RIGHT_HALF_CASE_DIR)
print("Output dir:", OUT_DIR)
print("Rollout checkpoint:", t.ROLLOUT_CHECKPOINT)
print("Case ids:", [spec["case_id"] for spec in CASE_SPECS])

## Generate Right-half Source Physical Case and Load Checkpoint


In [ ]:
required = [
    SK_CASE_DIR / "temperature_fields_3d_c.npy",
    SK_CASE_DIR / "temperature_3d_run_summary.json",
]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Missing South Kensington physical outputs. Run Run_South_Kensington_Temperature_3D_Labels.ipynb first. Missing:\n"
        + "\n".join(missing)
    )


def make_source_masks(shape):
    ydim, xdim = shape
    right_half = np.zeros((ydim, xdim), dtype=np.float32)
    right_half[:, xdim // 2 :] = 1.0
    return {
        "existing_source": np.ones((ydim, xdim), dtype=np.float32),
        "right_half_source": right_half,
    }


def apply_source_mask_to_boundary(boundary_fields, source_mask):
    out = {k: np.array(v, copy=True) for k, v in boundary_fields.items()}
    mask = source_mask.astype(np.float32)
    ambient0 = float(out["ambient_temp_series_c"][0])

    for key in ["ground_surface_temperature_excess_c", "building_surface_temperature_excess_c"]:
        if key in out:
            out[key] = (out[key].astype(np.float32) * mask).astype(np.float32)

    if "roof_surface_temperature_excess_3d" in out:
        out["roof_surface_temperature_excess_3d"] = (
            out["roof_surface_temperature_excess_3d"].astype(np.float32) * mask[None, :, :]
        ).astype(np.float32)

    if "ground_surface_temperature_excess_c" in out:
        out["ground_surface_temperature_c"] = (
            ambient0 + out["ground_surface_temperature_excess_c"]
        ).astype(np.float32)
    if "building_surface_temperature_excess_c" in out:
        out["building_surface_temperature_c"] = (
            ambient0 + out["building_surface_temperature_excess_c"]
        ).astype(np.float32)
    if "solid_temperature_3d" in out and "roof_surface_temperature_excess_3d" in out:
        solid = np.full_like(out["solid_temperature_3d"], ambient0, dtype=np.float32)
        roof_active = np.abs(out["roof_surface_temperature_excess_3d"]) > 1e-6
        solid[roof_active] = ambient0 + out["roof_surface_temperature_excess_3d"][roof_active]
        out["solid_temperature_3d"] = solid
    return out


def physical_case_complete(case_dir):
    required = [
        case_dir / "temperature_fields_3d_c.npy",
        case_dir / "temperature_fields_3d_masked_c.npy",
        case_dir / "temperature_3d_run_summary.json",
        case_dir / "source_test_metadata.json",
        case_dir / "source_mask.npy",
    ]
    return all(path.exists() and path.stat().st_size > 0 for path in required)


def source_case_matches_spec(case_dir, source_mask_name):
    if not physical_case_complete(case_dir):
        return False
    metadata = json.loads((case_dir / "source_test_metadata.json").read_text())
    return metadata.get("source_mask_name") == source_mask_name


velocity_config = SouthKensingtonConfig(
    timesteppings=50,
    model_resolution_m=4.0,
    z_dim=10,
    height_scale_m=4.0,
    output_dir=str(t.BUNDLE_ROOT / "outputs" / "south_kensington_velocity"),
)
right_half_temperature_config = Temperature3DScenarioConfig(
    output_dir=str(RIGHT_HALF_CASE_DIR),
    reuse_velocity_output_dir=str(t.BUNDLE_ROOT / "outputs" / "south_kensington_velocity_3d"),
    frame_duration_s=90.0,
    diffusion_coeff_m2_s=1.0,
    velocity_scale=1.0,
    save_animation=False,
    save_overview_figure=False,
    temperature_solver_backend="auto",
    temperature_solver_device="cuda",
    ambient_temp_c=24.2,
    inflow_temp_c=24.2,
    analysis_time_utc="2025-07-25T13:00:00+00:00",
    anthropogenic_heat_flux_w_m2=15.0,
)

if RUN_RIGHT_HALF_PHYSICAL_CASE:
    if (
        source_case_matches_spec(RIGHT_HALF_CASE_DIR, "right_half_source")
        and not FORCE_REGENERATE_RIGHT_HALF_PHYSICAL
    ):
        print("[skip] Existing right-half source physical case:", RIGHT_HALF_CASE_DIR)
    else:
        print("[run] Generating right-half source physical case:", RIGHT_HALF_CASE_DIR, flush=True)
        velocity_results, fields, cache_reused = extract_or_load_full_velocity_fields(
            velocity_config,
            right_half_temperature_config,
        )
        base_boundary_fields = build_surface_temperature_fields(
            fields,
            velocity_config,
            right_half_temperature_config,
        )
        source_mask = make_source_masks(fields["study_area_mask_2d"].shape)["right_half_source"]
        boundary_fields = apply_source_mask_to_boundary(base_boundary_fields, source_mask)
        temperature_results = solve_temperature_fields_3d(
            fields,
            boundary_fields,
            velocity_config,
            right_half_temperature_config,
        )
        summary = save_results(
            velocity_results,
            fields,
            boundary_fields,
            temperature_results,
            velocity_config,
            right_half_temperature_config,
            cache_reused,
        )
        np.save(RIGHT_HALF_CASE_DIR / "source_mask.npy", source_mask.astype(np.float32))
        metadata = {
            "case_id": "case_right_half_source",
            "source_mask_name": "right_half_source",
            "source_active_fraction": float(np.mean(source_mask[fields["study_area_mask_2d"]] > 0)),
            "note": "Only the right half keeps the existing source; the left half source excess is set to zero/ambient.",
            "velocity_config": asdict(velocity_config),
            "temperature_config": asdict(right_half_temperature_config),
        }
        (RIGHT_HALF_CASE_DIR / "source_test_metadata.json").write_text(
            json.dumps(metadata, indent=2)
        )
        print(
            json.dumps(
                {
                    "case_dir": str(RIGHT_HALF_CASE_DIR),
                    "source_active_fraction": metadata["source_active_fraction"],
                    "temperature_min_c": summary.get("temperature_min_c"),
                    "temperature_max_c": summary.get("temperature_max_c"),
                },
                indent=2,
            )
        )

checkpoint_path = t.ROLLOUT_CHECKPOINT if t.ROLLOUT_CHECKPOINT.exists() else t.FALLBACK_CHECKPOINT
if checkpoint_path == t.FALLBACK_CHECKPOINT:
    print("Rollout checkpoint not found; falling back to one-step checkpoint for comparison.")

checkpoint = torch.load(checkpoint_path, map_location="cpu")
config = checkpoint.get("config", {})
t.INFERENCE_MODE = t.recommended_inference_mode(config)
t.CORRECTION_ITERATIONS = int(config.get("correction_iterations", t.CORRECTION_ITERATIONS))
stats = t.SurrogateStats(**checkpoint["stats"])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = t.load_model_from_checkpoint(checkpoint, device)

cases = {
    spec["case_id"]: t.apply_z_crop(t.Temperature3DCase(spec["case_dir"])) for spec in CASE_SPECS
}
case = cases["case_baseline_existing_source"]
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
print("Loaded checkpoint:", checkpoint_path)
print("Checkpoint epoch:", checkpoint.get("epoch", "unknown"))
print("Z crop:", t.Z_CROP)
print("Inference mode:", t.INFERENCE_MODE)
for case_id, loaded_case in cases.items():
    print(f"{case_id} temperature shape:", loaded_case.temperature.shape)
    print(f"{case_id} velocity shape:", loaded_case.u.shape)

## Autoregressive Rollout for Both Cases


In [ ]:
rollout_results = {}
all_metrics = []
for spec in CASE_SPECS:
    case_id = spec["case_id"]
    print(f"[rollout] {case_id}", flush=True)
    case_obj = cases[case_id]
    pred_seq, true_seq, valid_mask, frame_ids = t.autoregressive_rollout(
        case_obj, model, stats, device, inference_mode=t.INFERENCE_MODE
    )
    err_seq = pred_seq - true_seq
    metrics = {
        "case_id": case_id,
        "checkpoint": str(checkpoint_path),
        "case_dir": str(spec["case_dir"]),
        "mode": "south_kensington_autoregressive_rollout",
        "source_mask_name": spec["source_mask_name"],
        "use_next_velocity": t.USE_NEXT_VELOCITY,
        "correction_iterations": t.CORRECTION_ITERATIONS,
        "z_crop": t.Z_CROP,
        "inference_mode": t.INFERENCE_MODE,
        "patch_size": list(t.PATCH_SIZE),
        "use_cool_surface_channels": t.USE_COOL_SURFACE_CHANNELS,
        "use_cool_surface_relaxation": t.USE_COOL_SURFACE_RELAXATION,
        "cool_surface_relaxation": t.COOL_SURFACE_RELAXATION,
        "open_ground_relaxation": t.OPEN_GROUND_RELAXATION,
        "cool_surface_vertical_decay_layers": t.COOL_SURFACE_VERTICAL_DECAY_LAYERS,
        "in_channels": t.input_channel_count(t.HISTORY_STEPS),
        "frames": [int(v) for v in frame_ids],
        "first_predicted_frame": int(frame_ids[0]),
        "first_step_bias_c": float(np.mean(err_seq[0][valid_mask])),
        "first_step_mae_c": float(np.mean(np.abs(err_seq[0][valid_mask]))),
        "first_step_rmse_c": float(np.sqrt(np.mean(err_seq[0][valid_mask] ** 2))),
        "final_frame": int(frame_ids[-1]),
        "final_bias_c": float(np.mean(err_seq[-1][valid_mask])),
        "final_mae_c": float(np.mean(np.abs(err_seq[-1][valid_mask]))),
        "final_rmse_c": float(np.sqrt(np.mean(err_seq[-1][valid_mask] ** 2))),
        "rollout_bias_c": float(np.mean(err_seq[:, valid_mask])),
        "rollout_mae_c": float(np.mean(np.abs(err_seq[:, valid_mask]))),
        "rollout_rmse_c": float(np.sqrt(np.mean(err_seq[:, valid_mask] ** 2))),
    }
    case_out_dir = OUT_DIR / case_id
    case_out_dir.mkdir(parents=True, exist_ok=True)
    (case_out_dir / "rollout_metrics.json").write_text(json.dumps(metrics, indent=2))
    np.save(case_out_dir / "pred_temperature_c.npy", pred_seq.astype(np.float32))
    np.save(case_out_dir / "true_temperature_c.npy", true_seq.astype(np.float32))
    np.save(case_out_dir / "valid_mask.npy", valid_mask.astype(bool))
    np.save(case_out_dir / "frame_ids.npy", frame_ids.astype(np.int32))
    rollout_results[case_id] = {
        "pred_seq": pred_seq,
        "true_seq": true_seq,
        "valid_mask": valid_mask,
        "frame_ids": frame_ids,
        "metrics": metrics,
        "case": case_obj,
        "out_dir": case_out_dir,
    }
    all_metrics.append(metrics)
    print(json.dumps(metrics, indent=2))

metrics_path = OUT_DIR / "south_kensington_rollout_metrics_all_cases.json"
metrics_path.write_text(json.dumps(all_metrics, indent=2))

# Backward-compatible aliases for the original baseline case.
pred_seq = rollout_results["case_baseline_existing_source"]["pred_seq"]
true_seq = rollout_results["case_baseline_existing_source"]["true_seq"]
valid_mask = rollout_results["case_baseline_existing_source"]["valid_mask"]
frame_ids = rollout_results["case_baseline_existing_source"]["frame_ids"]
err_seq = pred_seq - true_seq
print("Saved all-case metrics:", metrics_path)

## Final-frame k=6 Comparison for Both Cases


In [ ]:
K_VIS = 6
TEMP_LIMITS_C = (22.0, 32.0)

plot_rows = []
for case_id, result in rollout_results.items():
    case_obj = result["case"]
    velocity_dir = case_obj._velocity_cache_dir()
    building_mask_path = velocity_dir / "building_mask_2d.npy"
    building_mask_2d = (
        np.load(building_mask_path).astype(bool)
        if building_mask_path.exists()
        else np.any(case_obj.solid_mask, axis=0)
    )

    pred_final = result["pred_seq"][-1]
    true_final = result["true_seq"][-1]
    valid_mask_case = result["valid_mask"]
    frame_id = int(result["frame_ids"][-1])
    k_idx = min(K_VIS, pred_final.shape[0] - 1)
    err = pred_final - true_final
    slice_mask = valid_mask_case[k_idx]

    true_slice = true_final[k_idx].copy()
    pred_slice = pred_final[k_idx].copy()
    err_slice = err[k_idx].copy()
    true_slice[~slice_mask] = np.nan
    pred_slice[~slice_mask] = np.nan
    err_slice[~slice_mask] = np.nan
    slice_err = err[k_idx][slice_mask]

    plot_rows.append(
        {
            "case_id": case_id,
            "label": case_id.replace("case_", "").replace("_", " "),
            "frame_id": frame_id,
            "k_idx": int(k_idx),
            "true": true_slice,
            "pred": pred_slice,
            "err": err_slice,
            "building_mask": building_mask_2d,
            "bias": float(np.mean(slice_err)),
            "mae": float(np.mean(np.abs(slice_err))),
        }
    )

if not plot_rows:
    print("No rollout results available to visualize at k=6.")
else:
    temp_cmap = plt.get_cmap("YlOrRd").copy()
    temp_cmap.set_bad("white")
    err_cmap = plt.get_cmap("coolwarm").copy()
    err_cmap.set_bad("white")
    all_err_values = np.concatenate(
        [np.abs(row["err"][np.isfinite(row["err"])]).ravel() for row in plot_rows]
    )
    err_lim = max(float(np.percentile(all_err_values, 98)), 1e-6)

    fig, axes = plt.subplots(
        len(plot_rows), 3, figsize=(15, 4.3 * len(plot_rows)), constrained_layout=True
    )
    axes = np.atleast_2d(axes)
    for row_idx, row in enumerate(plot_rows):
        im0 = axes[row_idx, 0].imshow(
            row["true"],
            cmap=temp_cmap,
            vmin=TEMP_LIMITS_C[0],
            vmax=TEMP_LIMITS_C[1],
            origin="upper",
        )
        axes[row_idx, 0].set_title(
            f"{row['label']}\nPhysical model T | frame {row['frame_id']} | k={row['k_idx']}"
        )
        axes[row_idx, 1].imshow(
            row["pred"],
            cmap=temp_cmap,
            vmin=TEMP_LIMITS_C[0],
            vmax=TEMP_LIMITS_C[1],
            origin="upper",
        )
        axes[row_idx, 1].set_title(
            f"{row['label']}\nRollout-finetuned surrogate T | k={row['k_idx']}"
        )
        im2 = axes[row_idx, 2].imshow(
            row["err"], cmap=err_cmap, vmin=-err_lim, vmax=err_lim, origin="upper"
        )
        axes[row_idx, 2].set_title(
            f"Error: surrogate - physical\nbias={row['bias']:+.3f} C | MAE={row['mae']:.3f} C"
        )
        for ax in axes[row_idx]:
            ax.contour(
                row["building_mask"].astype(float), levels=[0.5], colors="black", linewidths=0.4
            )
            ax.set_xlabel("x index")
            ax.set_ylabel("y index")

    fig.colorbar(
        im0, ax=axes[:, :2], shrink=0.85, fraction=0.025, pad=0.02, label="Temperature (deg C)"
    )
    fig.colorbar(im2, ax=axes[:, 2], shrink=0.85, fraction=0.025, pad=0.02, label="deg C")
    fig.suptitle(
        f"South Kensington rollout-finetuned surrogate | final frame | k={K_VIS}", fontsize=14
    )
    k_panel_path = OUT_DIR / f"final_frame_k{K_VIS:02d}_both_cases.png"
    fig.savefig(k_panel_path, dpi=180, bbox_inches="tight")
    print("Saved k=6 comparison panel:", k_panel_path)
    plt.show()

## Error Through Time for Both Cases


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
for case_id, result in rollout_results.items():
    err_seq_case = result["pred_seq"] - result["true_seq"]
    valid_mask_case = result["valid_mask"]
    frame_ids_case = result["frame_ids"]
    frame_mae = np.asarray(
        [np.mean(np.abs(err_seq_case[i][valid_mask_case])) for i in range(err_seq_case.shape[0])],
        dtype=np.float32,
    )
    frame_bias = np.asarray(
        [np.mean(err_seq_case[i][valid_mask_case]) for i in range(err_seq_case.shape[0])],
        dtype=np.float32,
    )
    label = case_id.replace("case_", "").replace("_", " ")
    axes[0].plot(frame_ids_case, frame_mae, marker="o", label=label)
    axes[1].plot(frame_ids_case, frame_bias, marker="o", label=label)

axes[0].set_title("South Kensington rollout MAE")
axes[0].set_xlabel("frame")
axes[0].set_ylabel("deg C")
axes[0].grid(True, alpha=0.3)
axes[0].legend()
axes[1].axhline(0.0, color="black", linewidth=0.8)
axes[1].set_title("South Kensington rollout bias")
axes[1].set_xlabel("frame")
axes[1].set_ylabel("deg C")
axes[1].grid(True, alpha=0.3)
axes[1].legend()
plt.show()